# Setup

In [1]:
!pip install -qU openai pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.6/460.6 kB 6.1 MB/s eta 0:00:00


To jest instalacja dwóch bibliotek Pythona za pomocą menedżera pakietów pip:

`openai` - oficjalna biblioteka do komunikacji z API OpenAI, która umożliwia wykorzystanie modeli językowych i innych usług tej firmy w aplikacjach

`pydantic` - biblioteka do walidacji danych i serializacji w Pythonie, często używana do sprawdzania poprawności danych wejściowych i wyjściowych


In [ ]:
# Standard library imports
from typing import Literal

# Third-party imports
from openai import OpenAI
from pydantic import BaseModel
from google.colab import userdata


In [3]:
class CFG:
    max_new_tokens = 100
    model= 'gpt-4o-mini'

In [5]:
client = OpenAI(api_key = userdata.get('openaivision'))

# Dane

In [6]:
reviews = [
   "The room was clean and the staff was friendly.",
   "The location was terrible and the service was slow.",
   "The food was amazing but the room was too small.",
]

# Podstawowa ewaluacja

In [7]:
SYSTEM_PROMPT = "You are a sentiment classifier assistant."
PROMPT_TEMPLATE = """
   Classify the sentiment of the following hotel review as positive, negative, or neutral:\n\n{review}
"""

Ten fragment kodu definiuje dwa kluczowe elementy wykorzystywane w komunikacji z modelem językowym do analizy sentymentu recenzji hotelowych.

`SYSTEM_PROMPT` to instrukcja systemowa, która ustala kontekst i rolę modelu językowego. W tym przypadku krótka i zwięzła wiadomość "You are a sentiment classifier assistant" informuje model, że jego zadaniem jest klasyfikacja sentymentu. Jest to jak powiedzenie asystentowi: "Od tej pory jesteś specjalistą od określania wydźwięku emocjonalnego tekstów."

`PROMPT_TEMPLATE` to szablon zapytania, który będzie używany dla każdej recenzji. Jest to struktura, która zostanie wypełniona konkretną treścią recenzji w miejscu oznaczonym `{review}`. Szablon jasno określa zadanie: klasyfikację sentymentu recenzji hotelowej do jednej z trzech kategorii: pozytywnej, negatywnej lub neutralnej.

In [8]:
# normal eval
def classify_sentiment(review):
   response = client.beta.chat.completions.parse(
       model = CFG.model,
       messages=[
           {"role": "system", "content": SYSTEM_PROMPT},
           {"role": "user", "content": PROMPT_TEMPLATE.format(review=review)},
       ],
   )
   return response.choices[0].message

In [9]:
for review in reviews:
   sentiment = classify_sentiment(review)
   print(f"Review: {review}\nSentiment: {sentiment.content}\n")

Review: The room was clean and the staff was friendly.
Sentiment: Positive

Review: The location was terrible and the service was slow.
Sentiment: The sentiment of the hotel review is negative.

Review: The food was amazing but the room was too small.
Sentiment: The sentiment of the review is mixed, but it leans towards neutral due to the presence of both a positive aspect (the food) and a negative aspect (the room size). If you need a single classification, it could be considered neutral.



# Ewaluacja z SO

In [10]:
class SentimentResponse(BaseModel):
   sentiment: Literal["positive", "negative", "neutral"]

Ta definicja klasy `SentimentResponse` jest przykładem zastosowania systemu typów Pythona wraz z biblioteką Pydantic do stworzenia solidnej struktury danych dla odpowiedzi klasyfikatora sentymentu. Przyjrzyjmy się, jak działa ten kod i dlaczego jest tak użyteczny.

Klasa dziedziczy po `BaseModel` z biblioteki Pydantic, co nadaje jej szereg zaawansowanych możliwości walidacji danych. `BaseModel` to fundament, na którym Pydantic buduje swój system sprawdzania poprawności danych - zapewnia automatyczną walidację przy tworzeniu obiektów i serializację do różnych formatów.

Definicja pola `sentiment` z użyciem `Literal` to zaawansowana konstrukcja z modułu `typing`, która w tym przypadku ogranicza możliwe wartości sentymentu do dokładnie trzech opcji: "positive", "negative" lub "neutral". To ograniczenie działa jak zabezpieczenie - próba utworzenia obiektu `SentimentResponse` z jakąkolwiek inną wartością (na przykład "mixed" czy "very positive") spowoduje błąd walidacji.

Taka struktura przynosi szereg korzyści. Po pierwsze, zapewnia spójność danych - nie ma możliwości, aby w systemie pojawiły się nieoczekiwane wartości sentymentu. Po drugie, ułatwia dokumentację i współpracę w zespole - każdy programista może łatwo sprawdzić, jakie wartości są dozwolone. Po trzecie, pomaga w wykrywaniu błędów na wczesnym etapie - jeśli gdzieś w kodzie pojawi się próba użycia nieprawidłowej wartości, dowiemy się o tym natychmiast, a nie w momencie, gdy błędne dane dotrą do innej części systemu.


In [11]:
def classify_sentiment_with_structured_outputs(review):
   response = client.beta.chat.completions.parse(
       model= CFG.model,
       messages=[
           {"role": "system", "content": SYSTEM_PROMPT},
           {"role": "user", "content": PROMPT_TEMPLATE.format(review=review)},
       ],
       response_format=SentimentResponse
   )
   return response.choices[0].message


Ta funkcja `classify_sentiment_with_structured_outputs` jest zaawansowaną wersją wcześniejszego klasyfikatora sentymentu. Główna różnica polega na tym, że wymusza ona ściśle zdefiniowaną strukturę odpowiedzi, co znacząco zwiększa niezawodność i przewidywalność systemu.

Przyjrzyjmy się, jak funkcja została zaprojektowana. Podobnie jak w poprzedniej wersji, przyjmuje ona parametr `review` - tekst recenzji do analizy. Jednak sposób, w jaki przetwarza tę recenzję, jest bardziej wyrafinowany.

Kluczowa różnica znajduje się w wywołaniu metody `client.beta.chat.completions.parse()`. Do standardowych parametrów (`model` i `messages`) dodano nowy parametr: `response_format=SentimentResponse`. Jest to niezwykle istotne ulepszenie. Ten parametr mówi API, że oczekujemy odpowiedzi zgodnej ze strukturą zdefiniowaną w klasie `SentimentResponse`, którą wcześniej stworzyliśmy.

Przypomnijmy, że `SentimentResponse` pozwala tylko na trzy konkretne wartości: "positive", "negative" lub "neutral". Dzięki temu połączeniu uzyskujemy kilka znaczących korzyści:

1. Gwarancja poprawności danych - odpowiedź musi pasować do zdefiniowanego formatu, w przeciwnym razie system zgłosi błąd.
2. Automatyczna walidacja - nie musimy pisać dodatkowego kodu sprawdzającego poprawność odpowiedzi.
3. Lepsza dokumentacja - struktura odpowiedzi jest jasno określona w kodzie.
4. Łatwiejsze debugowanie - jeśli coś pójdzie nie tak, szybko dowiemy się, gdzie leży problem.

Ostatnia linia funkcji, `return response.choices[0].message`, pozostaje taka sama, ale teraz mamy pewność, że zwracana wiadomość będzie zawierała dokładnie jeden z trzech dozwolonych sentymentów.

To ulepszenie jest przykładem zasady "fail fast" w programowaniu - lepiej jest wykryć problem jak najwcześniej, niż pozwolić niepoprawnym danym przemieszczać się przez system. Jest to szczególnie ważne w kontekście pracy z modelami AI, gdzie odpowiedzi mogą być czasami nieprzewidywalne.

In [ ]:
for review in reviews:
    sentiment = classify_sentiment_with_structured_outputs(review)
    print(f"Review: {review}\nSentiment: {sentiment.content}")
    print('-' * 10)

Review: The room was clean and the staff was friendly.
Sentiment: {"sentiment":"positive"}
----------
Review: The location was terrible and the service was slow.
Sentiment: {"sentiment":"negative"}
----------
Review: The food was amazing but the room was too small.
Sentiment: {"sentiment":"neutral"}
----------
